# Data Inventory — IBM Telco Customer Churn

Profile the source dataset before designing the schema.

Establish what the data contains before making any modelling decisions: dimensions, column types, missing values, and the reliability of supplied fields. Use the findings here to determine the schema and feature set built in the SQL layer.

**Scope note:** treat this notebook as exploratory only. Implement all joins, aggregation, feature engineering and CLV logic in SQL. Carry no transformation from this notebook into the modelling pipeline.

---

## 1. Retrieve the dataset

Download the IBM Telco Customer Churn dataset (full version) via `kagglehub`. Use the full version rather than the more common abridged release — it includes the `CLTV`, `Churn Reason` and geographic fields required for the customer value layer.

In [ ]:
import kagglehub, os

path = kagglehub.dataset_download("yeanzc/telco-customer-churn-ibm-dataset")
print("Path:", path)
print(os.listdir(path))

c:\Users\IC Clearwater\OneDrive\Documents\GitHub\Customer_Lifetime_Value_Churn_Prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 1.25M/1.25M [00:01<00:00, 680kB/s]

Extracting files...


Path: C:\Users\IC Clearwater\.cache\kagglehub\datasets\yeanzc\telco-customer-churn-ibm-dataset\versions\1
['Telco_customer_churn.xlsx']


## 2. Load the file and confirm its structure

Load the workbook. Confirm the record count and list the columns.

Check first that `CLTV` is present, as the customer value segmentation depends on it. Confirm this before doing any further work rather than assuming the dataset's contents.

In [ ]:
import pandas as pd

df = pd.read_excel(f"{path}/Telco_customer_churn.xlsx")
print(df.shape)
print(df.columns.tolist())
pd.set_option('display.max_columns', None)
df.head(10)

(7043, 33)
['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value', 'Churn Score', 'CLTV', 'Churn Reason']


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices
5,4190-MFLUW,1,United States,California,Los Angeles,90020,"34.066367, -118.309868",34.066367,-118.309868,Female,No,Yes,No,10,Yes,No,DSL,No,No,Yes,Yes,No,No,Month-to-month,No,Credit card (automatic),55.20,528.35,Yes,1,78,5925,Competitor offered higher download speeds
6,8779-QRDMV,1,United States,California,Los Angeles,90022,"34.02381, -118.156582",34.023810,-118.156582,Male,Yes,No,No,1,No,No phone service,DSL,No,No,Yes,No,No,Yes,Month-to-month,Yes,Electronic check,39.65,39.65,Yes,1,100,5433,Competitor offered more data
7,1066-JKSGK,1,United States,California,Los Angeles,90024,"34.066303, -118.435479",34.066303,-118.435479,Male,No,No,No,1,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,20.15,20.15,Yes,1,92,4832,Competitor made better offer
8,6467-CHFZW,1,United States,California,Los Angeles,90028,"34.099869, -118.326843",34.099869,-118.326843,Male,No,Yes,Yes,47,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.35,4749.15,Yes,1,77,5789,Competitor had better devices
9,8665-UTDHZ,1,United States,California,Los Angeles,90029,"34.089953, -118.294824",34.089953,-118.294824,Male,No,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,No,Electronic check,30.20,30.2,Yes,1,97,2915,Competitor had better devices


## 3. Review column types and missing values

Print the data type of each column. Count the nulls, filtering to columns where nulls are present.

Check the data types — a numeric field stored as text signals a data quality issue that must be corrected before use.

In [ ]:
pd.set_option('display.max_rows', 40)
print(df.dtypes)
print("\nNulls:")
print(df.isna().sum()[df.isna().sum() > 0])

CustomerID               str
Count                  int64
Country                  str
State                    str
City                     str
Zip Code               int64
Lat Long                 str
Latitude             float64
Longitude            float64
Gender                   str
Senior Citizen           str
Partner                  str
Dependents               str
Tenure Months          int64
Phone Service            str
Multiple Lines           str
Internet Service         str
Online Security          str
Online Backup            str
Device Protection        str
Tech Support             str
Streaming TV             str
Streaming Movies         str
Contract                 str
Paperless Billing        str
Payment Method           str
Monthly Charges      float64
Total Charges         object
Churn Label              str
Churn Value            int64
Churn Score            int64
CLTV                   int64
Churn Reason             str
dtype: object

Nulls:
Churn Reason    5174


**Result:** `Churn Reason` is null for 5,174 records. The dataset holds 7,043 customers, of whom 1,869 churned, leaving 5,174 retained. The nulls therefore correspond exactly to customers who have not churned and for whom no reason exists. Expected behaviour — apply no treatment.

## 4. Inspect the truncated columns

The previous output was truncated in display. Print the final ten columns separately to complete the type inventory.

In [ ]:
print(df.dtypes.tail(10))

Contract                 str
Paperless Billing        str
Payment Method           str
Monthly Charges      float64
Total Charges         object
Churn Label              str
Churn Value            int64
Churn Score            int64
CLTV                   int64
Churn Reason             str
dtype: object


**Result:** `Monthly Charges` is stored as `float64` but `Total Charges` is stored as `object`, i.e. text. Both fields hold currency values and should share a numeric type. This points to non-numeric entries inside `Total Charges`.

## 5. Investigate the non-numeric entries in Total Charges

Do not coerce the column directly. Isolate the affected records first and examine them to establish the cause.

Apply `errors='coerce'` to convert any unparseable value into a null. Use those nulls to identify the problem rows, then inspect them against tenure and churn status.

In [ ]:
bad = pd.to_numeric(df['Total Charges'], errors='coerce').isna()
print(bad.sum())
print(df.loc[bad, ['Tenure Months', 'Monthly Charges', 'Total Charges', 'Churn Label']])

11
      Tenure Months  Monthly Charges Total Charges Churn Label
2234              0            52.55                        No
2438              0            20.25                        No
2568              0            80.85                        No
2667              0            25.75                        No
2856              0            56.05                        No
4331              0            19.85                        No
4687              0            25.35                        No
5104              0            20.00                        No
5719              0            19.70                        No
6772              0            73.35                        No
6840              0            61.90                        No


**Result:** 11 records affected. All show `Tenure Months` of zero and all are active customers. These are newly acquired accounts not yet billed, so no total charge exists to record.

The blank is accurate rather than corrupt, so substitute zero. Retain the records — deleting them would remove valid new-customer observations from the population.

## 6. Correct Total Charges and summarise the numeric fields

Convert `Total Charges` to numeric, substituting zero for the eleven unbilled accounts. Then produce descriptive statistics for the core numeric fields.

In [ ]:
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce').fillna(0)
print(df['Total Charges'].dtype)
print(df[['Tenure Months','Monthly Charges','Total Charges','CLTV']].describe())

float64
       Tenure Months  Monthly Charges  Total Charges         CLTV
count    7043.000000      7043.000000    7043.000000  7043.000000
mean       32.371149        64.761692    2279.734304  4400.295755
std        24.559481        30.090047    2266.794470  1183.057152
min         0.000000        18.250000       0.000000  2003.000000
25%         9.000000        35.500000     398.550000  3469.000000
50%        29.000000        70.350000    1394.550000  4527.000000
75%        55.000000        89.850000    3786.600000  5380.500000
max        72.000000       118.750000    8684.800000  6500.000000


**Result:** `Total Charges` now returns `float64` and is available for calculation.

The summary raises a concern over `CLTV`. The field is bounded between 2,003 and 6,500, a range inconsistent with a genuine distribution of customer value. `Total Charges` meanwhile reaches 8,684 — revenue already collected. A lifetime value lower than revenue already received is not possible for a currency-denominated measure.

## 7. Test the reliability of the supplied CLTV field

Treat lifetime value as a function of periodic revenue and expected tenure. A valid measure must therefore correlate strongly with `Monthly Charges`, `Total Charges` and `Tenure Months`.

Use Spearman rank correlation rather than Pearson. The question is whether `CLTV` orders customers consistently with its own inputs, independent of scale.

In [ ]:
print(df[['CLTV','Monthly Charges','Total Charges','Tenure Months','Churn Value']].corr(method='spearman')['CLTV'])

CLTV               1.000000
Monthly Charges    0.107944
Total Charges      0.310721
Tenure Months      0.367039
Churn Value       -0.123627
Name: CLTV, dtype: float64


**Result:** correlations fall materially below what the measure requires.

| Field | Spearman ρ with CLTV | Expected |
|---|---|---|
| Total Charges | 0.31 | Strong positive |
| Tenure Months | 0.37 | Strong positive |
| Monthly Charges | 0.11 | Strong positive |
| Churn Value | -0.12 | Clearly non-zero |

`CLTV` does not track the variables that define lifetime value. Read alongside the bounded range identified above, the field is a synthetic score rather than a derived financial measure.

**Decision:** do not use the supplied `CLTV` field. Calculate customer lifetime value independently from `Monthly Charges`, `Tenure Months` and a stated cost-to-serve assumption, and document the derivation in full. Do not validate against `CLTV` — it provides no reliable reference.

---

## Findings

1. **Dataset dimensions** — 7,043 customer records across 33 fields.

2. **Missing values** — confined to `Churn Reason` and fully explained by churn status. No treatment required.

3. **Data quality** — `Total Charges` held blanks for 11 unbilled new accounts, forcing the column to text. Corrected to numeric with zero substituted.

4. **Supplied CLTV rejected** — the field is bounded, falls below revenue already collected for some customers, and shows negligible correlation with its own determinants. Derive CLV independently.

5. **Target leakage identified** — exclude `Churn Score` and `Churn Reason` from the feature set. `Churn Score` is a model output shipped with the dataset; `Churn Reason` is populated only after churn has occurred. Either would leak the target and inflate performance.

**Next stage:** design the relational schema and implement feature engineering in SQL.